# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a thorough step-by-step guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its Croissant schema URL:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Suppress warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Explore metadata: name and description
meta = dataset.metadata
print(f"\033[1m{meta.name}\033[0m")
print(meta.description)
print(f'Identifier: {meta.identifier}')
print(f'Published: {meta.datePublished}')
print(f'Version: {meta.version}')
print(f'Authors (IDs): {meta.author}')

## 2. Data Overview
Review available record sets, their `@id`s, fields, and columns.

**Note:** In Croissant, *record sets* are the tabular data resources. Fields (table columns) are referenced using their unique `@id`s.

Let's enumerate the available record sets, their fields, and all corresponding `@id`s.

In [ ]:
from mlcroissant import helpers

# Show all record sets and their fields/columns, referencing by @id

record_sets = dataset.metadata.recordSet
if not record_sets:
    raise ValueError('No record sets found in this dataset metadata.')

overview = []
print('Record Sets Overview:')
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}  (name: {rs.get('name', '<unnamed>')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Number of fields: {len(fields)}")
    for f in fields:
        print(f"    Field @id: {f['@id']:50} | name: {f.get('name', '<unnamed>')}")
        cols = f.get('column', [])
        if isinstance(cols, dict):
            cols = [cols]
        for c in cols:
            print(f"      Column @id: {c['@id']:50} | name: {c.get('name', '<unnamed>')}")

## 3. Data Extraction
Load the data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

> **Important:** All access to record sets, fields, and columns is performed via their `@id`.

We'll extract all record sets, display the fields of one as an example, and show a preview.

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

dfs = {}
for rec_id in record_set_ids:
    print(f'Loading records from record set: {rec_id}')
    try:
        records = list(dataset.records(record_set=rec_id))
        if records:
            dfs[rec_id] = pd.DataFrame(records)
            print(f'  -> Loaded {len(dfs[rec_id])} records [{list(dfs[rec_id].columns)}]')
        else:
            print('  -> No records found for this record set.')
    except Exception as e:
        print(f'  ! Skipping due to error: {e}')

# Choose the first non-empty dataframe for demonstration
main_rs_id = None
for rec_id, df in dfs.items():
    if not df.empty:
        main_rs_id = rec_id
        break

if main_rs_id is None:
    raise RuntimeError('No record sets with records found.')

print(f"\nColumns in record set '{main_rs_id}':")
print(dfs[main_rs_id].columns.tolist())
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing a numeric field, and grouping by a categorical field.

Let's select a likely numeric field (e.g., age) and categorize by another attribute. All operations reference field columns via their `@id` as per Croissant.

In [ ]:
# Pick a numeric field (using exact @id as shown earlier, e.g., 'cr:age' or similar)
# You may adjust these based on the actual overview from above
# For demonstration, let's assume an age column is '@id': 'http://senscience.ai/field/age'
# and a group field, for example, 'http://senscience.ai/field/sex' (Sex)

# Try to find a likely numeric field and group field from columns
num_field_candidates = [col for col in dfs[main_rs_id].columns if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()]
group_field_candidates = [col for col in dfs[main_rs_id].columns if 'sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower() or 'group' in col.lower()]

if not num_field_candidates:
    raise ValueError('No likely numeric field (e.g., age) found.')
if not group_field_candidates:
    print('No obvious categorical field for grouping found, proceeding without grouping.')

numeric_field_id = num_field_candidates[0]
group_field_id = group_field_candidates[0] if group_field_candidates else None

# Ensure the numeric field is numeric for EDA
df = dfs[main_rs_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering, e.g., age > 50
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by the categorical field, if any
if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean '{numeric_field_id}' grouped by '{group_field_id}':")
    print(grouped)
else:
    print('Grouping skipped (no categorical field identified).')

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using `matplotlib` and/or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load metadata and records via a Croissant schema using `mlcroissant`.
- Enumerate record sets and reference all dataset entities by their `@id`.
- Extract and preview data from tabular record sets.
- Perform basic EDA with numeric filtering, normalization, and grouping.
- Visualize variable distributions and relationships between key fields.

All dataset access has referenced entities by their `@id` as required for robust, schema-valid Croissant workflows.

### Useful references
- [mlcroissant documentation](https://mlcommons.github.io/croissant/tools/mlcroissant/)
- [FAIR² dataset source](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)